In [5]:
reachable_airports(search_params=search_params, top_airports=['BEG'])

In [3]:
nearest_city((48.86284945224139, 2.3372743394481112))

In [2]:
search_params = {
    "id": 1250015082,
    "start_date_string":"2025-04-30",
    "end_date_string":"2025-05-21",
    "activity_tag":"summer"
}
nearby_airports(search_params=search_params)
#
results = get_result(search_params)


In [5]:
results[2]

In [127]:
direct_flight_destinations = results[2]
all_flight_destinations = results[3]

In [135]:
[
    (
        flight['departure']['iataCode'],
        flight['departure']['scheduledTime'],
        flight['arrival']['iataCode'],
        flight['arrival']['scheduledTime'],
        flight['airline']['name']
    ) for flight in all_flight_destinations['CNF']['departures']]

In [130]:
allowed_airports = direct_flight_destinations['airports'].to_list()




In [138]:

from itertools import chain


def include_ae(all_flight_destinations, direct_flight_destinations):
    to_destination_airports = {}
    for key in all_flight_destinations.keys():
        iata_codes = jmespath.search(f"{key}.departures[*].arrival.iataCode", all_flight_destinations)
        to_destination_airports.update({key:iata_codes})
        to_destination_airports.update({key:list(set(to_destination_airports[key]))})

    from_destination_airports = {}
    for key in all_flight_destinations.keys():
        iata_codes = jmespath.search(f"{key}.arrivals[*].departure.iataCode", all_flight_destinations)
        from_destination_airports.update({key:iata_codes})
        from_destination_airports.update({key:list(set(from_destination_airports[key]))})

    to_destination_airports = [to_destination_airports[key] for key in to_destination_airports.keys()]
    from_destination_airports = [from_destination_airports[key] for key in from_destination_airports.keys()]
    all_from_origin = list(set(list(chain.from_iterable(to_destination_airports))))
    all_to_origin = list(set(list(chain.from_iterable(from_destination_airports))))
    conditions = [pl.col('airports').list.contains(airport.upper()) for airport in all_from_origin]
    df_new = direct_flight_destinations.with_columns(
        pl.reduce(lambda a, b: a | b, conditions).alias('from destination'),
    )
    conditions = [pl.col('airports').list.contains(airport.upper()) for airport in all_to_origin]
    df_new = df_new.with_columns(
        pl.reduce(lambda a, b: a | b, conditions).alias('to destination'))
    result = df_new.filter((pl.col('to destination')==True)&(pl.col('from destination')==True))
    return result

In [140]:
include_ae(all_flight_destinations=all_flight_destinations, direct_flight_destinations=direct_flight_destinations)

In [137]:
df_new

In [120]:
conditions = [pl.col('airports').list.contains(airport.upper()) for airport in all_from_origin]
df_new = direct_flight_destinations.with_columns(
    pl.reduce(lambda a, b: a | b, conditions).alias('from destination'),
)
conditions = [pl.col('airports').list.contains(airport.upper()) for airport in all_to_origin]
df_new = df_new.with_columns(
    pl.reduce(lambda a, b: a | b, conditions).alias('to destination'))

In [126]:
df_new.filter(pl.col('from destination')==True)['airports'].to_list()


In [122]:
df_new.filter((pl.col('a2')==True))

In [15]:
all_flight_destinations['BEG']['departures']

In [ ]:
departure_iata_codes

In [ ]:
all

In [ ]:
all_flight_destinations

In [ ]:
all

In [ ]:
all_flight_destinations

In [ ]:
def extract_keys(d, target_keys):
    if isinstance(d, dict):
        for k, v in d.items():
            if k in target_keys:
                yield (k, v)
            yield from extract_keys(v, target_keys)
    elif isinstance(d, list):
        for item in d:
            yield from extract_keys(item, target_keys)

target_keys = {"iata", "passengers", "city"}

results = list(extract_keys(all_flight_destinations, target_keys))


In [ ]:
autocomplete_cities.filter(pl.col('city_ascii')=='Belgrade')

In [ ]:

search_params = {
    "id": 1688374696,
    "start_date_string":"2025-05-12",
    "end_date_string":"2025-05-27",
    "activity_tag":"summer"
}

top_airports = nearby_airports(search_params, large_airports=1, airports=3)['iata_code'].to_list()

In [ ]:

all_flights = reachable_airports(search_params=search_params,top_airports=top_airports)

In [ ]:

search_params = {
    "id": 1688374696,
    "start_date_string":"2024-12-12",
    "end_date_string":"2024-12-27",
    "activity_tag":"summer"
}

top_airports = nearby_airports(search_params, large_airports=5, airports=10)['iata_code'].to_list()
import requests
from concurrent import futures

def reachable_airports(search_params, top_airports):

    departure_date = search_params['start_date_string']
    return_date = search_params['end_date_string']


    def call_API_destinations(url):
        response = requests.get(url)
        return response.json()

    # Generate URLs for each departure location (2 per location)
    list_bodies = []
    for loc in top_airports:
        list_bodies.append(
            f'https://aviation-edge.com/v2/public/flightsFuture?iataCode={loc}&type=departure&date={departure_date}&key={aviation_edge_key}'
        )
        list_bodies.append(
            f'https://aviation-edge.com/v2/public/flightsFuture?iataCode={loc}&type=arrival&date={return_date}&key={aviation_edge_key}'
        )

    results = []
    with futures.ThreadPoolExecutor(max_workers=len(list_bodies)) as executor:
        for result in executor.map(call_API_destinations, list_bodies):
            results.append(result)

    # Organize results by location
    structured = {}
    for idx, loc in enumerate(top_airports):
        structured[loc] = {
            "departures": results[idx * 2],
            "arrivals": results[idx * 2 + 1]
        }

    return structured


In [ ]:
# reference DS
tag_name = "peaks"
relevant_feature_list = ['PK','MT']
h3_feature_hotel_distance = (5,1)
h3_feature_city_distance = (5,1)
h3_city_hotel_distance = (6,0)
h3_city_airport_distance = (3,1)
min_hotels_cities = 0
min_hotels_features = 0
min_pop = -1
foc_min_pop=50000
max_pop=50000000
h3_data_dir="countries_h3"
countries_metadata = pl.read_parquet('raw_files/countries_metadata.parquet')
fca_folder = "parquet"
output_dir = 'countries_filtered'
activity_tag_root_dir = 'activity_tags_raw_data'

In [ ]:
make_activity_tag(tag_name=tag_name,countries_metadata=countries_metadata,h3_data_dir=h3_data_dir, relevant_feature_list=relevant_feature_list, h3_feature_hotel_distance=h3_feature_hotel_distance,h3_feature_city_distance=h3_feature_city_distance,h3_city_hotel_distance=h3_city_hotel_distance,h3_city_airport_distance=h3_city_airport_distance,min_hotels_cities=min_hotels_cities, min_hotels_features=min_hotels_features,min_pop=min_pop,foc_min_pop=foc_min_pop, max_pop=max_pop,fca_folder = fca_folder)

In [ ]:
tag = 'summer'
cities = pl.read_parquet(f'parquet/{tag}/cities.parquet')
airports = pl.read_parquet(f'parquet/{tag}/airports.parquet')
features = pl.read_parquet(f'parquet/{tag}/features.parquet')
with open(f"parquet/{tag}/metadata.json","r") as c:
    metadata = json.load(c)
score_matrix = pl.read_parquet(f'parquet/{tag}/score_matrix.parquet')
with open(f"parquet/{tag}/weights.json","r") as c:
    weights = json.load(c)
features_list = pl.read_parquet('raw_files/featureCodes.parquet')

In [ ]:
airports_s = pl.read_csv('airports_new.csv')
airports_h3 = add_h3_index(df = airports_s, lat_col='latitude_deg', lon_col='longitude_deg')

In [ ]:
coordinates = 44.80276425444967, 20.449912715854772
start_point = nearest_city(coordinates=coordinates)

In [ ]:
start_point

In [ ]:
new = pl.read_csv('airports_new.csv')

In [ ]:
new_filter = new.filter((pl.col('scheduled_service')=='yes')&(pl.col('type')!='closed')&(pl.col('type')!='heliport')&(pl.col('type')!='seaplane_base'))

In [ ]:
a1 = new_filter.filter(pl.col('type').is_in(['large_airport','medium_airport']))
a1.write_parquet('raw_files/airports_filtered.parquet')

In [ ]:
a1

In [ ]:
h3.grid_disk(start_point["h3_index_4"][0],1)

In [ ]:
features.filter(pl.col(f'h3_index_4').is_in(h3.grid_disk(start_point[f"h3_index_4",1])))


In [ ]:
start_date_string = "2024-12-12"
end_date_string = "2024-12-31"
start_date = datetime.strptime(start_date_string, "%Y-%m-%d")
end_date = datetime.strptime(end_date_string, "%Y-%m-%d")
trip_duration = (end_date - start_date).days
trip_duration

In [ ]:
h3_radius = {
    '0'	:1281.256011,
    '1'	:483.0568391,
    '2'	:182.5129565,
    '3'	:68.97922179,
    '4'	:26.07175968,
    '5'	:9.854090990,
    '6'	:3.724532667
}

In [ ]:
autocomplete_cities.filter(pl.col('city_ascii')=='Belgrade')['h3_index_0'][0]

In [ ]:
import requests
from concurrent import futures
aviation_edge_key = '1b69d4-e620ae'
user_request = {'departureLocation': 'BEG',
                'departureDate' : '2025-04-06',
                'returnDate' :'2025-05-15'
                }

def reachable_airports(user_request):

    departure_location = user_request['departureLocation']
    departure_date = user_request['departureDate']
    return_date = user_request['returnDate']


    def call_API_destinations(body):
        response = requests.get(body)
        return response.json()

    list_bodies = [
        f'https://aviation-edge.com/v2/public/flightsFuture?iataCode={departure_location}&type=departure&date={departure_date}&key={aviation_edge_key}',
        f'https://aviation-edge.com/v2/public/flightsFuture?iataCode={departure_location}&type=arrival&date={return_date}&key={aviation_edge_key}'
    ]
    num_treads = len(list_bodies)
    results = []
    with futures.ThreadPoolExecutor(max_workers=num_treads) as executor:
        for i in executor.map(call_API_destinations, list_bodies):
            results.append(i)

    return {
        'departures': results[0],
        'arrivals': results[1],
            }

In [ ]:
results = reachable_airports(user_request=user_request)

In [ ]:
autocomplete_cities

In [ ]:
metadata

In [ ]:

def search_area(trip_duration):
    if trip_duration == 0:
        daily_radius = (4,0)
        flight_radius = None
        direct_flight_radius = (1,0)
    elif trip_duration == 1:
        daily_radius = (4,1)
        flight_radius = None
        direct_flight_radius = (1,0)
        starting_airport_radius = (4,0)
    elif trip_duration == 2:
        daily_radius = (2,0)
        flight_radius = (1,1)
        direct_flight_radius = (1,1)
        starting_airport_radius = (4,0)
    elif trip_duration == 3:
        daily_radius = (2,0)
        flight_radius = (1,1)
        direct_flight_radius = (1,1)
        starting_airport_radius = (4,0)
    elif trip_duration == 4:
        daily_radius = (2,0)
        flight_radius = (1,1)
        direct_flight_radius = (1,1)
        starting_airport_radius = (4,0)
    elif trip_duration == 5:
        daily_radius = (2,0)
        flight_radius = (2,0)
        direct_flight_radius = (1,1)
        starting_airport_radius = (4,0)
    elif 5<trip_duration<7:
        daily_radius = (2,0)
        flight_radius = (0,1)
        direct_flight_radius = (0,1)
        starting_airport_radius = (4,1)
    elif 7<=trip_duration <=10:
        daily_radius = (1,1)
        flight_radius = (0,2)
        direct_flight_radius = (0,2)
        starting_airport_radius = (3,1)
    else:
        daily_radius = (0,1)
        flight_radius = 'all'
        direct_flight_radius = 'all'
        starting_airport_radius = (3,1)
    return daily_radius, flight_radius, direct_flight_radius, starting_airport_radius

In [ ]:
airports_all = pl.read_parquet(f'raw_files/airports_h3.parquet')

In [ ]:
airports_all

In [ ]:
airports_all = pl.read_parquet(f'raw_files/airports_filtered_h3.parquet')

def add_h3_index_polars(df, lat_col,lon_col):

    all_indexes = []
    for j in range(8):
        h3_index = []
        for row in df.to_dicts():
            h3_index.append(h3.latlng_to_cell(row[lat_col], row[lon_col],j))
        all_indexes.append(h3_index)
    for i in range(8):
        df = df.with_columns([
        pl.Series(f"h3_index_{i}", all_indexes[i]),
    ])
    return df

add_h3_index_polars(df=airports_all, lat_col='latitude_deg', lon_col='longitude_deg')


In [ ]:
for i in range(8):
    airports_all = airports_all.with_columns([
    pl.Series(f"h3_index_{i}", all_indexes[i]),
])

In [ ]:
airports_all.write_parquet('raw_files/airports_filtered_h3.parquet')

In [ ]:

airports_all.write_parquet('raw_files/airports_h3.parquet')

In [ ]:
airports_all.filter(pl.col('country_code')=='US')

In [ ]:
autocomplete_cities.filter(pl.col('city_ascii')==''

In [ ]:
autocomplete_cities.filter(pl.col('city')=='New York')

In [ ]:
import haversine as hs
def nearby_airports(search_params,airports = 3, large_airports = 1, autocomplete_cities = autocomplete_cities, all_airports =all_airports):
    try:
        start_date = datetime.strptime(search_params["start_date_string"], "%Y-%m-%d")
        end_date = datetime.strptime(search_params["end_date_string"], "%Y-%m-%d")
        trip_duration = (end_date - start_date).days
        start_point = autocomplete_cities.filter(pl.col('id')==search_params["id"])


        search_area_params = search_area(trip_duration)
        start_airport_radius = search_area_params[3]
        index_start_airports = start_point[f'h3_index_{start_airport_radius[0]}'][0]
        nearby_airports = all_airports.filter(pl.col(f'h3_index_{start_airport_radius[0]}').is_in(h3.grid_disk(index_start_airports,start_airport_radius[1])))
        for index in reversed(range(start_airport_radius[0],6)):
            for radius in range(start_airport_radius[1]+1):
                print(index,radius)
                relevant_nearby_airports = nearby_airports.filter(pl.col(f'h3_index_{index}').is_in(h3.grid_disk(start_point[f'h3_index_{index}'][0],radius)))
                relevant_nearby_airports = relevant_nearby_airports.sort(by=pl.col("type") != "large_airport")
                if (relevant_nearby_airports.height>=airports and relevant_nearby_airports.filter(pl.col('type')=='large_airport').height>=large_airports):

                    break
        coordinates  = start_point['lat'][0],start_point['lng'][0]
        found_list = list(zip(relevant_nearby_airports["latitude_deg"], relevant_nearby_airports["longitude_deg"]))
        distances = hs.haversine_vector(coordinates,found_list, comb=True).flatten().tolist()
        relevant_nearby_airports = relevant_nearby_airports.with_columns(pl.Series("distance", distances)).sort(by="distance", descending=False).head(3)
    except:
        relevant_nearby_airports = 'no_airports_found'

    return relevant_nearby_airports





search_params = {
    "id": 1840034016,
    "start_date_string":"2024-12-12",
    "end_date_string":"2024-12-15",
    "activity_tag":"summer"
}

nearby_airports(search_params)



In [ ]:
search_params = {
    "id": 1840034016,
    "start_date_string":"2024-12-12",
    "end_date_string":"2024-12-31",
    "activity_tag":"summer"
}
def get_result(search_params, autocomplete_cities = autocomplete_cities, all_airports = all_airports):
    start_date = datetime.strptime(search_params["start_date_string"], "%Y-%m-%d")
    end_date = datetime.strptime(search_params["end_date_string"], "%Y-%m-%d")
    trip_duration = (end_date - start_date).days
    start_point = autocomplete_cities.filter(pl.col('id')==search_params["id"])


    search_area_params = search_area(trip_duration)
    daily_trip = search_area_params[0]
    flight_trip = search_area_params[1]
    direct_flight_trip = search_area_params[2]
    start_airport_radius = search_area_params[3]

    #importing relevant data
    cities = pl.read_parquet(f'parquet/{search_params["activity_tag"]}/cities.parquet')
    airports = pl.read_parquet(f'parquet/{search_params["activity_tag"]}/airports.parquet')
    with open(f'parquet/{search_params["activity_tag"]}/metadata.json','r') as c:
        metadata = json.load(c)
    score_matrix = pl.read_parquet(f'parquet/{search_params["activity_tag"]}/score_matrix.parquet')
    with open(f'parquet/{search_params["activity_tag"]}/weights.json',"r") as c:
        weights = json.load(c)
    features = pl.read_parquet(f'parquet/{search_params["activity_tag"]}/features.parquet')

    index_start_airports = start_point[f'h3_index_{start_airport_radius[0]}'][0]
    nearby_airports = all_airports.filter(pl.col(f'h3_index_{start_airport_radius[0]}').is_in(h3.grid_disk(index_start_airports,start_airport_radius[1])))
    scored_cities = get_scored_cities(cities=cities,score_matrix=score_matrix,weights=weights,airports=airports,metadata=metadata)
    car_trip_results = features.filter(pl.col(f'h3_index_{daily_trip[0]}').is_in(h3.grid_disk(start_point[f"h3_index_{daily_trip[0]}"][0],daily_trip[1])))

    if flight_trip != None:
        if flight_trip =='all':
            flight_trip_results = scored_cities
        else:
            flight_trip_results = scored_cities.filter(pl.col(f'h3_index_{flight_trip[0]}').is_in(h3.grid_disk(start_point[f'h3_index_{flight_trip[0]}'][0],flight_trip[1])))
    else:
        flight_trip_results = 'no results'

    if direct_flight_trip == 'all':
        direct_flight_trip_results = scored_cities
    else:
        direct_flight_trip_results = scored_cities.filter(pl.col(f'h3_index_{direct_flight_trip[0]}').is_in(h3.grid_disk(start_point[f'h3_index_{direct_flight_trip[0]}'][0],direct_flight_trip[1])))


    return car_trip_results, flight_trip_results, direct_flight_trip_results, nearby_airports


In [ ]:
results = get_result(search_params, autocomplete_cities )
results[3]

In [ ]:
us_a = pl.read_parquet('countries_filtered/US/airports_filtered.parquet')

In [ ]:
us_a

In [ ]:
airports.filter(pl.col('h3_index_3').is_in(h3.grid_disk('831e1bfffffffff',1)))

In [ ]:
results[3]

In [ ]:
results[1]

In [ ]:
results[1].filter(pl.col('country_code').is_in(['IT','ES','ME','HR','FR','TR','EG'])).group_by(pl.col('country_code')).head(10)

In [ ]:
nearest_city_result = nearest_city((23.788120288071966, 79.21992622609741), autocomplete_cities=autocomplete_cities)

In [ ]:
nearest_city_result

In [ ]:
result = get_scored_cities(cities, score_matrix, weights, airports, metadata)

In [ ]:
nearest_city_result

In [ ]:
result

In [ ]:
autocomplete_cities

In [ ]:
def nearest_city(coordinates,autocomplete_cities):
    allowed_indexes = [(6,0),(6,1),(5,0),(5,1),(4,0),(4,1),(3,0),(3,1)]
    for h3_index_radius in allowed_indexes:
        autocomplete_city_index = h3.latlng_to_cell(coordinates[0],coordinates[1],h3_index_radius[0])
        grid_disk = h3.grid_disk(autocomplete_city_index,h3_index_radius[1])
        found_cities = autocomplete_cities.filter(pl.col(f'h3_index_{h3_index_radius[0]}').is_in(grid_disk))
        if found_cities.height!=0:
            break

        else:
            pass

    found_list = list(zip(found_cities["lat"], found_cities["lng"]))
    distances = hs.haversine_vector(coordinates,found_list, comb=True).flatten().tolist()
    found_cities =found_cities.with_columns(pl.Series("distance", distances)).sort(by="distance", descending=False).head(1)

    return found_cities

In [ ]:
nearest_city((23.788120288071966, 79.21992622609741))

In [ ]:
grid_disk

In [ ]:
result = get_scored_cities(cities, score_matrix, weights, airports, metadata)
result_dict=result.head(50).select(result.columns[0:5]).to_dicts()
with open("mock_result.json", "w",encoding="utf-8") as f:
    json.dump(result_dict, f, indent=4, ensure_ascii=False)

In [ ]:
result1=result.head(50).select(result.columns[0:5]).to_dicts()

In [ ]:
result

In [ ]:
result1

In [ ]:
with open("mock_result.json", "w",encoding="utf-8") as f:
    json.dump(result1, f, indent=4, ensure_ascii=False)

In [ ]:
with open("mock_result.json", "r", encoding="utf-8") as f:
    x = json.load(f)
x

In [ ]:
result.filter(pl.col('country_code')=='ME')

In [ ]:
all_indexes = h3.grid_disk('851ef213fffffff',1)
features.filter(pl.col('h3_index_5').is_in(all_indexes)).filter(pl.col('feature_code')=='BAY')